# FourCastNet Colab Smoke Test (Tensor-First)

## ⚠️ Advertencia de Validez Científica (Premortem)
Este notebook tiene como objetivo validar la **ejecución mecánica** del modelo FourCastNet v0. 
Un paso exitoso (forward pass) no garantiza validez meteorológica si los canales `sp`, `tcwv` o `r` fueron sintetizados mediante proxies.

**Estado Actual del Contrato:**
- **Orden de Canales:** NVlabs v0 Backbone Official (0..19).
- **Política de Stats:** `first_20_channels` (SST/Canal 20 ignorado).
- **Formato:** Inferencia directa desde tensor `.npy` (HDF5 opcional).

## 1. Configuración de Entorno y Montaje de Drive
Cargue el archivo `fourcastnet_colab_bundle.zip` a su Google Drive y especifique la ruta a continuación.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# EDITAR ESTA RUTA según la ubicación en su Drive
BUNDLE_PATH = '/content/drive/MyDrive/fcn_colab_bundle.zip' 
WORK_DIR = '/content/fcn_run'

if os.path.exists(BUNDLE_PATH):
    !mkdir -p {WORK_DIR}
    !unzip -q {BUNDLE_PATH} -d {WORK_DIR}
    print(f'✅ Bundle extraído en {WORK_DIR}')
    os.chdir(WORK_DIR)
else:
    print(f'❌ ERROR: No se encontró el bundle en {BUNDLE_PATH}')

## 2. Pre-vuelo de Diagnóstico (Preflight Diagnostics)
Este paso detecta errores de normalización o corrupción de datos antes de usar la GPU.

In [ ]:
!pip -q install numpy pandas matplotlib torch timm pyyaml

if os.path.exists('colab_preflight_checks.py'):
    !python colab_preflight_checks.py
    from IPython.display import Image, display
    print("\n--- Verificación de Alineación de Stats ---")
    display(Image('zscore_alignment_check.png'))
    print("\n--- Verificación Visual de Tensores (T2M, Z1000, T500) ---")
    display(Image('input_tensor_visual_check.png'))
else:
    print('⚠️ Script de pre-vuelo no encontrado.')

## 3. Inicialización de Reporte e Inferencia
Definición estricta de canales según el contrato NVlabs v0.

In [ ]:
import sys
import json
import shutil
import traceback
import importlib
import numpy as np
import torch
import os

EXPECTED_SHAPE = (1, 20, 720, 1440)
CHANNELS = [
    ('u10', None), ('v10', None), ('t2m', None), ('sp', None), ('msl', None),
    ('t', 850.0), ('u', 1000.0), ('v', 1000.0), ('z', 1000.0), ('u', 850.0),
    ('v', 850.0), ('z', 850.0), ('u', 500.0), ('v', 500.0), ('z', 500.0),
    ('t', 500.0), ('z', 50.0), ('r', 500.0), ('r', 850.0), ('tcwv', None),
]

report = {
    'tensor_load_ok': False,
    'normalization_ok': False,
    'inference_success': False,
    'failure_reason': None
}

def run_inference():
    try:
        # 1. Carga de Tensor e Inicialización
        x = np.load('input_tensor.npy').astype(np.float32)
        if x.shape != EXPECTED_SHAPE:
            raise ValueError(f'Shape incorrecto: {x.shape}')
        report['tensor_load_ok'] = True

        # 2. Normalización (Política first_20_channels)
        m = np.load('global_means.npy').astype(np.float32)
        s = np.load('global_stds.npy').astype(np.float32)

        # Slicing explícito para asegurar 20 canales si el archivo tiene 21 (SST)
        m = m[:, :20, :, :] if m.ndim == 4 else m[:20].reshape(1, 20, 1, 1)
        s = s[:, :20, :, :] if s.ndim == 4 else s[:20].reshape(1, 20, 1, 1)

        x_norm = (x - m) / s
        report['normalization_ok'] = bool(np.isfinite(x_norm).all())

        # 3. Clonar repositorio y cargar modelo
        if not os.path.exists('/content/FourCastNet'):
            os.system('git clone -q https://github.com/NVlabs/FourCastNet /content/FourCastNet')
        
        # Patch numpy pad import in afnonet.py for newer numpy versions
        os.system("sed -i 's/from numpy.lib.arraypad import pad/from numpy import pad/g' /content/FourCastNet/networks/afnonet.py")
        # Instalar ruamel.yaml requerido por YParams
        os.system('pip install -q ruamel.yaml')

        sys.path.insert(0, '/content/FourCastNet')
        sys.path.insert(0, '/content/FourCastNet/networks')

        from networks.afnonet import AFNONet
        from utils.YParams import YParams
        
        params = YParams('/content/FourCastNet/config/AFNO.yaml', 'afno_backbone')
        params.N_in_channels = 20
        params.N_out_channels = 20
        model = AFNONet(params, img_size=(720, 1440), in_chans=20, out_chans=20)

    # --- SEGURIDAD DE PRODUCCIÓN (LIMPIEZA DE PESOS) ---
        ckpt_path = 'backbone.ckpt'
        clean_path = 'clean_backbone_weights.pt'
        
        if not os.path.exists(clean_path):
            print('🧹 Extrayendo state_dict limpio para futuras cargas seguras...')
            # Usamos weights_only=False una única vez para extraer los datos
            raw_ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state_dict = raw_ckpt['model_state'] if 'model_state' in raw_ckpt else raw_ckpt
            torch.save(state_dict, clean_path)
            print(f'✅ Pesos guardados en {clean_path}')
        
        # Carga final segura (weights_only=True)
        state_dict = torch.load(clean_path, map_location='cpu', weights_only=True)
        model.load_state_dict(state_dict, strict=False)
        model.eval()

        # 4. Inferencia en CPU (Smoke Test)
        with torch.no_grad():
            xt = torch.from_numpy(x_norm)
            y_norm = model(xt)

        report['inference_success'] = bool(np.isfinite(y_norm.cpu().numpy()).all())
        print('✅ Inferencia completada con éxito.')

    except Exception as e:
        report['failure_reason'] = str(e)
        print(f'❌ ERROR: {e}')
        traceback.print_exc()

run_inference()
